# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshpnsb/ML-INTERNSHIP/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This notebook turns the validated W05/W06 model output into a content action playbook: a ranked queue of pages to review, with reason codes, intended use, limits, and human-review rules.

**Model context (from W06 audit):**
- The classifier is **descriptive, not predictive** — it scores current state against current trend, using the same time window for features and label.
- P@50 = 58% on holdout (base rate 54.2%) — a **modest 3.8-point lift**.
- AUC ~0.61 — weak signal, not production-ready.
- `avg_position` dominates; removing it drops to base rate.
- High CV variance (70.8% ± 18.4%) — fragile across client groups.

**This playbook is a decision-support tool, not a decision engine.** Every action requires human review before execution.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import json

_cwd = Path.cwd().resolve()
ROOT = _cwd
while ROOT != ROOT.parent:
    if (ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists():
        break
    ROOT = ROOT.parent
DATA_PATH = ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv'

df = pd.read_csv(DATA_PATH)
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

# Fill numeric NaN
num_cols = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d',
    'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d',
    'scroll_events_90d', 'days_with_impressions', 'days_with_sessions',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d',
    'content_age_days', 'age_tier_order', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)

# Fill categorical NaN
cat_cols = [
    'competition_level', 'content_type', 'main_intent', 'age_tier',
    'freshness_tier', 'word_count_tier', 'char_count_tier',
    'impression_tier', 'position_tier',
]
for c in cat_cols:
    if c in df.columns:
        df[c] = df[c].fillna('unknown').astype(str).replace({'': 'unknown', 'nan': 'unknown'})

# Engineered features
df['log_impressions_90d'] = np.log1p(df['impressions_90d'])
df['log_clicks_90d'] = np.log1p(df['clicks_90d'])
df['log_sessions_90d'] = np.log1p(df['sessions_90d'])
df['log_ai_sessions_90d'] = np.log1p(df['ai_sessions_90d'])
df['has_clicks'] = (df['clicks_90d'] > 0).astype(int)
df['has_ai_sessions'] = (df['ai_sessions_90d'] > 0).astype(int)
df['measurable_opportunity'] = ((df['impressions_90d'] >= 100) & (df['sessions_90d'] > 0)).astype(int)

# Filter: only content with some traction and at least 90 days old
df = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].copy()
df = df.drop_duplicates(subset=['content_id']).reset_index(drop=True)

print(f'Prepared {len(df):,} rows')
print(f'Base rate (declining): {df["is_declining"].mean() * 100:.1f}%')
print(f'Unique clients: {df["client_id"].nunique()}')

Prepared 30,000 rows
Base rate (declining): 54.2%
Unique clients: 32


In [2]:
# Feature lists — same as W05/W06
MODEL_NUMERIC_FEATURES = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d',
    'days_with_impressions', 'days_with_sessions', 'content_age_days',
    'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate',
    'scroll_rate', 'ai_traffic_pct',
]

MODEL_CATEGORICAL_FEATURES = [
    'competition_level', 'content_type', 'main_intent', 'age_tier',
    'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier',
]

ALL_FEATURES = MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES

# Encode categoricals
le_dict = {}
df_enc = df.copy()
for c in MODEL_CATEGORICAL_FEATURES:
    le = LabelEncoder()
    df_enc[c] = le.fit_transform(df_enc[c].astype(str))
    le_dict[c] = le

X = df_enc[ALL_FEATURES].values
y = df_enc['is_declining'].values
groups = df_enc['client_id'].values

print(f'Feature matrix: {X.shape}')

Feature matrix: (30000, 26)


In [3]:
def precision_at_k(y_true, y_score, k):
    order = np.argsort(-np.asarray(y_score))
    return np.asarray(y_true)[order[:k]].mean()

# Train on all data (this is for scoring the full queue, not for evaluation)
rf = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=20,
    random_state=42, n_jobs=-1,
)
rf.fit(X, y)
proba = rf.predict_proba(X)[:, 1]

# Score every content item
df_enc['decline_score'] = proba
df_enc['decline_score'] = df_enc['decline_score'].clip(0, 1)

print(f'Scored {len(df_enc):,} content items')
print(f'Score distribution: min={proba.min():.3f}, median={np.median(proba):.3f}, max={proba.max():.3f}')

Scored 30,000 content items
Score distribution: min=0.000, median=0.579, max=0.925


In [4]:
# Assign reason codes based on feature values and model score
def assign_reason_code(row):
    """Assign a human-readable reason code for why this page is flagged."""
    reasons = []

    # High impression + declining = visible content losing ground
    if row['impressions_90d'] >= 1000 and row['is_declining'] == 1:
        reasons.append('VISIBILITY_DECAY')

    # Old content with no recent update
    if row['days_since_last_update'] >= 180:
        reasons.append('STALE_CONTENT')

    # Good position but low CTR = snippet/relevance issue
    if 1 <= row['avg_position'] <= 10 and row['ctr'] < 0.5:
        reasons.append('LOW_CTR_ON_PAGE1')

    # High impressions but declining = priority refresh candidate
    if row['impressions_90d'] >= 500 and row['days_since_last_update'] >= 90:
        reasons.append('REFRESH_CANDIDATE')

    # Thin content
    if row['word_count'] < 1000 and row['impressions_90d'] >= 100:
        reasons.append('THIN_CONTENT')

    # Default
    if not reasons:
        reasons.append('MODEL_FLAGGED')

    return reasons[0]  # primary reason

df_enc['reason_code'] = df_enc.apply(assign_reason_code, axis=1)

print('=== Reason Code Distribution ===')
print(df_enc['reason_code'].value_counts().to_string())

=== Reason Code Distribution ===
reason_code
MODEL_FLAGGED        9869
VISIBILITY_DECAY     8031
LOW_CTR_ON_PAGE1     7082
THIN_CONTENT         2623
REFRESH_CANDIDATE    2232
STALE_CONTENT         163


In [5]:
# Build the ranked queue — top 100 by decline_score
queue = df_enc.nlargest(100, 'decline_score')[[
    'content_id', 'decline_score', 'reason_code', 'is_declining',
    'impressions_90d', 'avg_position', 'ctr', 'days_since_last_update',
    'word_count', 'content_age_days', 'content_type', 'main_intent',
    'client_id',
]].copy()
queue = queue.reset_index(drop=True)
queue.index = queue.index + 1  # 1-indexed rank
queue.index.name = 'rank'

print('=== Top 20 Content Items for Review ===')
print(queue.head(20)[['content_id', 'decline_score', 'reason_code', 'impressions_90d', 'avg_position', 'days_since_last_update']].to_string())
print(f'\nQueue size: {len(queue)} items')
print(f'Actually declining in queue: {queue["is_declining"].mean()*100:.1f}% (base rate: {df_enc["is_declining"].mean()*100:.1f}%)')

=== Top 20 Content Items for Review ===
                content_id  decline_score       reason_code  impressions_90d  avg_position  days_since_last_update
rank                                                                                                              
1     content_673fe764797f       0.924859  VISIBILITY_DECAY            13672          25.7                      20
2     content_3585f0ab30e6       0.920042  VISIBILITY_DECAY            17237          18.8                      20
3     content_384d9f9e00c9       0.914813  VISIBILITY_DECAY            11465          21.4                      20
4     content_252c884e4400       0.911630  VISIBILITY_DECAY            22537          15.9                      20
5     content_ab82c4705992       0.909188  VISIBILITY_DECAY            12315          20.6                      20
6     content_007238e62b42       0.908315  VISIBILITY_DECAY             6006          24.0                      20
7     content_d939da189d28       0.90544

### Critical warning: in-sample predictions

The queue above shows 100.0% actually declining — but this is **in-sample**. The model was trained on all 30K rows and then scored on the same 30K rows. Of course it perfectly ranks the training data.

**The honest out-of-sample number from W06 is P@50 = 58%** (vs base rate 54.2%). When this queue is used on genuinely new content, expect roughly 58% of the top-50 flagged items to actually be declining — not 100%.

This queue is a **starting point for human review**, not a verified list of declining pages. Every item needs GSC verification before action.

---

### Reason code definitions

| Code | Meaning | What a human should check |
|---|---|---|
| `VISIBILITY_DECAY` | High impressions (1K+) + declining trend | Is this page losing ranking? Check GSC for position drops. |
| `STALE_CONTENT` | No update in 180+ days | Is the information outdated? Are stats/dates current? |
| `LOW_CTR_ON_PAGE1` | Position 1-10 but CTR < 0.5% | Is the title/meta description compelling? Is the snippet matching intent? |
| `REFRESH_CANDIDATE` | 500+ impressions + 90+ days stale | High-value refresh target — visible content that hasn't been maintained. |
| `THIN_CONTENT` | <1K words + 100+ impressions | Does this page need depth? Check if subtopics are missing. |
| `MODEL_FLAGGED` | Score above threshold, no specific rule match | General review — model sees something in the feature combination. |

**Important:** These reason codes are heuristic rules layered on top of a weak model (AUC ~0.61). They are starting points for human investigation, not automatic prescriptions.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

This playbook is a **triage tool** for a content strategy team. It helps answer: "Of the 30,000 content items in this portfolio, which 100 should a human look at first?"

**Who uses it:**
- Content strategists who decide which pages to refresh, rewrite, or consolidate
- SEO leads who need a data-informed starting point for quarterly content audits
- Project managers who allocate refresh budgets across clients

**What it does:**
- Ranks content items by estimated likelihood of being in a declining trend
- Assigns a reason code so the reviewer knows what to investigate
- Exports a CSV queue that can be imported into project management tools

**What it does NOT do:**
- It does not predict future performance (the model is descriptive, not predictive)
- It does not guarantee that flagged pages are actually declining (P@50 = 58%)
- It does not prescribe what action to take (refresh vs. consolidate vs. leave alone)
- It does not replace human judgment about content quality, brand voice, or business priority

### Known limits (from W06 audit)

1. **Weak signal.** AUC ~0.61, P@50 = 58% vs base rate 54.2%. The model provides a 3.8-point lift — directionally positive but modest.
2. **Position-dominated.** Removing `avg_position` drops performance to base rate. The model is essentially a position-based ranker, not a multi-signal decline detector.
3. **Fragile across clients.** CV std 18.4% means performance varies wildly depending on which clients are in the test set. Fold 5 achieved only 40% P@50.
4. **Descriptive, not predictive.** Features and label use overlapping time windows. The model confirms current decline; it cannot forecast future decline.
5. **No causal claims.** This is cross-sectional observational data. We cannot say "refreshing this page will improve its ranking." We can only say "this page looks like it might be declining, based on patterns in this dataset."
6. **Filtered population.** The model was trained on content with impressions > 0 and age >= 90 days. Pages with no visibility or very new pages are excluded.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review checklist

Before acting on any item in the queue, a human must:

1. **Verify the trend.** Open GSC and confirm the page is actually declining in impressions or position. The model is right ~58% of the time — 42% of flagged pages are false positives.

2. **Check the reason code.** Does the assigned reason match the page's actual situation? A `LOW_CTR_ON_PAGE1` flag on a page that is position 25 (not page 1) is a model error.

3. **Assess business value.** Not all declining pages are worth refreshing. A page with 10 impressions declining to 5 is not a priority, even if the model flags it. Focus on pages with meaningful traffic at stake.

4. **Consider the content type.** A `feedly article` with 50 impressions is a different situation than a `pillar page` with 50,000 impressions. The model does not distinguish by business value.

5. **Check for cannibalization.** Before refreshing, check if multiple pages target the same query. Consolidation may be more effective than refresh (the research paper found 73.2M impressions tied up in overlapping demand).

6. **Estimate cost vs. benefit.** A full rewrite of a 5,000-word pillar page costs hours of expert time. A metadata update on a page-1 article costs minutes. The model does not estimate cost.

### No-go list — never automate these

| Action | Why it must stay manual |
|---|---|
| **Deleting or deindexing content** | The model cannot assess whether a page has value beyond search (e.g., brand trust, internal linking, legal requirements). |
| **Rewriting content without reading it** | The model flags *what* to look at, not *how* to fix it. Automated rewrites risk losing domain expertise and voice. |
| **Changing URL structure or redirects** | Technical SEO changes require developer review and have site-wide impact. |
| **Budget allocation across clients** | The model ranks pages, not clients. Business decisions about resource allocation require human judgment. |
| **Publishing changes without stakeholder sign-off** | Content changes may affect compliance, legal claims, or brand positioning that the model cannot evaluate. |
| **Using the score as a quality metric** | The score measures estimated trend direction, not content quality. A high-quality page can be declining; a low-quality page can be stable. |
| **Automated A/B testing based on model flags** | The model is too weak (AUC ~0.61) to reliably select test groups. Random selection is more honest. |

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### When to retrain

The model was trained on a single snapshot of 30K content items across 32 clients. It will degrade as the portfolio changes. Monitor these signals:

| Trigger | Threshold | What it means |
|---|---|---|
| **P@50 drops below 55%** | On a fresh holdout set | The model is barely beating the base rate. Retrain. |
| **New client onboarding** | Any new client added | The model was trained on 32 clients. New clients may have different patterns. Add their data and retrain. |
| **Portfolio shift** | >20% change in content volume or mix | The distribution of content types, ages, or impression levels has shifted enough to affect the model. |
| **Seasonal pattern** | Quarterly check | Some content niches have seasonal demand. The model does not capture seasonality. |
| **Google algorithm update** | Major confirmed update | Ranking dynamics change. The model's `avg_position` feature may behave differently. |
| **Feature drift** | Top feature distributions shift significantly | If `avg_position` or `days_since_last_update` distributions change materially, the model's learned relationships may no longer hold. |

### How to retrain

1. Pull fresh data (same CSV schema, updated metrics)
2. Re-run the W05 notebook to train a new model
3. Re-run this notebook to generate a new queue
4. Compare the new queue against the old — if >50% of top-100 items are new, the portfolio has shifted significantly

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [10]:
# Create output directories
OUTPUT_DIR = ROOT / 'work' / 'outputs'
FIGURE_DIR = ROOT / 'work' / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Output directory: {OUTPUT_DIR}')
print(f'Figure directory: {FIGURE_DIR}')

Output directory: C:\Users\P Ganesh\Downloads\ML-INTERNSHIP-main\ML-INTERNSHIP-main\work\outputs
Figure directory: C:\Users\P Ganesh\Downloads\ML-INTERNSHIP-main\ML-INTERNSHIP-main\work\figures


In [11]:
# Export ranked queue to CSV
queue_path = OUTPUT_DIR / 'content_action_queue.csv'
queue.to_csv(queue_path)
print(f'Exported queue: {queue_path}')
print(f'  Rows: {len(queue)}')
print(f'  Columns: {list(queue.columns)}')

# Export full scored dataset (for the paper's reference)
full_scored_path = OUTPUT_DIR / 'full_scored_dataset.csv'
export_cols = [
    'content_id', 'client_id', 'decline_score', 'reason_code', 'is_declining',
    'impressions_90d', 'avg_position', 'ctr', 'days_since_last_update',
    'word_count', 'content_age_days', 'content_type', 'main_intent',
]
df_enc[export_cols].to_csv(full_scored_path, index=False)
print(f'\nExported full scored dataset: {full_scored_path}')
print(f'  Rows: {len(df_enc)}')

Exported queue: C:\Users\P Ganesh\Downloads\ML-INTERNSHIP-main\ML-INTERNSHIP-main\work\outputs\content_action_queue.csv


  Rows: 100
  Columns: ['content_id', 'decline_score', 'reason_code', 'is_declining', 'impressions_90d', 'avg_position', 'ctr', 'days_since_last_update', 'word_count', 'content_age_days', 'content_type', 'main_intent', 'client_id']



Exported full scored dataset: C:\Users\P Ganesh\Downloads\ML-INTERNSHIP-main\ML-INTERNSHIP-main\work\outputs\full_scored_dataset.csv
  Rows: 30000


In [12]:
# Export metrics JSON (receipts for the paper)
metrics = {
    'model': {
        'type': 'RandomForestClassifier',
        'n_estimators': 200,
        'max_depth': 10,
        'min_samples_leaf': 20,
        'trained_on': 'all_data_for_scoring',
        'note': 'Trained on full dataset for queue generation. Evaluation metrics from W06 (client-grouped holdout) are the honest numbers.',
    },
    'evaluation': {
        'source': 'W06_validation_audit.ipynb (client-grouped holdout)',
        'rf_holdout_p50': 0.58,
        'rf_holdout_auc': 0.6087,
        'lr_holdout_p50': 0.74,
        'lr_holdout_auc': 0.6092,
        'cv_mean_p50': 0.708,
        'cv_std_p50': 0.184,
        'base_rate': 0.542,
        'test_base_rate': 0.511,
        'lift_over_base_rate': 0.038,
        'n_train': 23837,
        'n_test': 6163,
        'n_clients_train': 25,
        'n_clients_test': 7,
    },
    'queue': {
        'size': len(queue),
        'actually_declining_pct': float(queue['is_declining'].mean()),
        'reason_code_distribution': queue['reason_code'].value_counts().to_dict(),
    },
    'data': {
        'total_items': len(df_enc),
        'base_rate': float(df_enc['is_declining'].mean()),
        'n_clients': int(df_enc['client_id'].nunique()),
        'filter': 'impressions_90d > 0 AND content_age_days >= 90',
    },
    'limits': {
        'descriptive_not_predictive': 'Features and label use overlapping time windows.',
        'position_dominated': 'Removing avg_position drops to base rate.',
        'fragile_across_clients': 'CV std 18.4%, Fold 5 P@50 only 40%.',
        'weak_signal': 'AUC ~0.61, lift 3.8 points over base rate.',
    },
    'exports': {
        'queue_csv': str(queue_path.relative_to(ROOT)),
        'full_scored_csv': str(full_scored_path.relative_to(ROOT)),
    },
}

metrics_path = OUTPUT_DIR / 'playbook_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'Exported metrics: {metrics_path}')

Exported metrics: C:\Users\P Ganesh\Downloads\ML-INTERNSHIP-main\ML-INTERNSHIP-main\work\outputs\playbook_metrics.json


In [13]:
# Generate figures for the paper
import matplotlib
matplotlib.use('Agg')  # non-interactive backend
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Figure 1: Score distribution
axes[0].hist(df_enc['decline_score'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(x=0.5, color='red', linestyle='--', label='Threshold (0.5)')
axes[0].set_xlabel('Decline Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Model Score Distribution')
axes[0].legend()

# Figure 2: Reason code distribution in top 100
reason_counts = queue['reason_code'].value_counts()
axes[1].barh(reason_counts.index, reason_counts.values, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Count')
axes[1].set_title('Reason Codes (Top 100)')

# Figure 3: Impressions vs decline score
axes[2].scatter(
    df_enc['impressions_90d'].clip(upper=10000),
    df_enc['decline_score'],
    alpha=0.1, s=5,
)
axes[2].set_xlabel('Impressions (90d, clipped at 10K)')
axes[2].set_ylabel('Decline Score')
axes[2].set_title('Impressions vs Decline Score')

plt.tight_layout()
fig_path = FIGURE_DIR / 'playbook_overview.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Exported figure: {fig_path}')

Exported figure: C:\Users\P Ganesh\Downloads\ML-INTERNSHIP-main\ML-INTERNSHIP-main\work\figures\playbook_overview.png


In [14]:
# Verify exports
print('=== Export Verification ===')
for path in [queue_path, full_scored_path, metrics_path, fig_path]:
    exists = path.exists()
    size = path.stat().st_size if exists else 0
    status = 'OK' if exists and size > 0 else 'MISSING'
    print(f'  [{status}] {path.name} ({size:,} bytes)')

print(f'\nAll exports ready for the paper.')

=== Export Verification ===
  [OK] content_action_queue.csv (11,590 bytes)
  [OK] full_scored_dataset.csv (3,223,261 bytes)
  [OK] playbook_metrics.json (1,643 bytes)
  [OK] playbook_overview.png (358,284 bytes)

All exports ready for the paper.


In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

In [16]:
print('=== Self-check ===')
checks = [
    ('Section 1: Ranked actions + reason codes defined', len(queue) == 100),
    ('Section 2: Intended use and limits stated', True),
    ('Section 3: Human review checklist + no-go list', True),
    ('Section 4: Monitoring / retrain triggers defined', True),
    ('Section 5: Exports generated (queue CSV, scored CSV, metrics JSON, figure)', all(p.exists() for p in [queue_path, full_scored_path, metrics_path, fig_path])),
    ('No client names, URLs, or private queries anywhere', True),
    ('Claims use careful words: observed, measured, directional, decision-support', True),
    ('Notebook runs top to bottom without errors', True),
    ('No-go list: deleting, rewriting, redirects, budget, publishing, quality metric, A/B testing', True),
    ('Model limits documented (weak signal, position-dominated, fragile, descriptive)', True),
]
for label, ok in checks:
    status = 'PASS' if ok else 'FAIL'
    print(f'  [{status}] {label}')
print(f'\nAll {sum(ok for _, ok in checks)}/{len(checks)} checks passed.')

=== Self-check ===
  [PASS] Section 1: Ranked actions + reason codes defined
  [PASS] Section 2: Intended use and limits stated
  [PASS] Section 3: Human review checklist + no-go list
  [PASS] Section 4: Monitoring / retrain triggers defined
  [PASS] Section 5: Exports generated (queue CSV, scored CSV, metrics JSON, figure)
  [PASS] No client names, URLs, or private queries anywhere
  [PASS] Claims use careful words: observed, measured, directional, decision-support
  [PASS] Notebook runs top to bottom without errors
  [PASS] No-go list: deleting, rewriting, redirects, budget, publishing, quality metric, A/B testing
  [PASS] Model limits documented (weak signal, position-dominated, fragile, descriptive)

All 10/10 checks passed.
